In [ ]:
!pip install duckdb pandas numpy scikit-learn

In [ ]:
import duckdb
import pandas as pd
import numpy as np

con = duckdb.connect()

con.execute("""
CREATE SECRET (
    TYPE huggingface,
    TOKEN 
)
""")

rel = "hf://datasets/FlyRank/internship-warehouse"

print("Connected successfully!")

Connected successfully!


In [ ]:
con.sql(f"""
SELECT COUNT(*)
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,count_star()
0,78835655


In [ ]:

con.sql(f"""
SELECT *
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
LIMIT 5
""").df().columns.tolist()

['report_date',
 'client_hash_id',
 'content_hash_id',
 'client_has_gsc',
 'client_has_ga4',
 'gsc_data_available',
 'ga4_data_available',
 'gsc_impressions',
 'gsc_clicks',
 'gsc_sum_position',
 'gsc_avg_position',
 'ga4_pageviews',
 'ga4_sessions',
 'ga4_users',
 'ga4_engaged_sessions',
 'ga4_total_engagement_sec',
 'sessions_organic',
 'sessions_direct',
 'sessions_referral',
 'sessions_social',
 'sessions_paid',
 'sessions_ai',
 'ai_chatgpt',
 'ai_perplexity',
 'ai_gemini',
 'ai_copilot',
 'ai_claude',
 'ai_meta',
 'ai_other',
 'scroll_events',
 'month']

In [ ]:
df = con.sql(f"""
SELECT
    content_hash_id,

    SUM(gsc_impressions) AS total_impressions,
    SUM(gsc_clicks) AS total_clicks,
    AVG(gsc_avg_position) AS avg_position,

    SUM(ga4_pageviews) AS total_pageviews,
    SUM(ga4_sessions) AS total_sessions,
    SUM(ga4_users) AS total_users,
    SUM(ga4_engaged_sessions) AS total_engaged_sessions,
    SUM(ga4_total_engagement_sec) AS total_engagement_time,

    SUM(sessions_organic) AS organic_sessions,
    SUM(sessions_direct) AS direct_sessions,
    SUM(sessions_referral) AS referral_sessions,
    SUM(sessions_social) AS social_sessions,
    SUM(sessions_ai) AS ai_sessions,

    SUM(scroll_events) AS total_scroll_events

FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)

GROUP BY content_hash_id

LIMIT 50000
""").df()

print("Dataset shape:", df.shape)

df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Dataset shape: (50000, 15)


,content_hash_id,total_impressions,total_clicks,avg_position,total_pageviews,total_sessions,total_users,total_engaged_sessions,total_engagement_time,organic_sessions,direct_sessions,referral_sessions,social_sessions,ai_sessions,total_scroll_events
0,content_d5479697a0828b33,6115.0,31.0,33.292638,22.0,20.0,20.0,3.0,204.0,6.0,3.0,0.0,0.0,0.0,9.0
1,content_d7daac9d28863fda,12182.0,65.0,14.182897,82.0,70.0,68.0,2.0,713.0,44.0,32.0,0.0,0.0,0.0,40.0
2,content_b313be479d6d3707,16870.0,130.0,12.537416,59.0,57.0,57.0,1.0,256.0,55.0,7.0,0.0,0.0,0.0,25.0
3,content_5175438fecb054a4,21688.0,147.0,12.718705,15.0,15.0,14.0,2.0,386.0,11.0,6.0,0.0,0.0,0.0,4.0
4,content_3a204f16288e29ed,2535.0,8.0,20.989052,10.0,10.0,10.0,0.0,2.0,5.0,4.0,0.0,0.0,0.0,7.0


In [ ]:
print(df.isnull().sum())

content_hash_id             0
total_impressions           0
total_clicks                0
avg_position              646
total_pageviews             0
total_sessions              0
total_users                 0
total_engaged_sessions      0
total_engagement_time       0
organic_sessions            0
direct_sessions             0
referral_sessions           0
social_sessions             0
ai_sessions                 0
total_scroll_events         0
dtype: int64


In [ ]:
df["avg_position"] = df["avg_position"].fillna(
    df["avg_position"].median()
)

numeric_columns = df.select_dtypes(
    include=[np.number]
).columns

df[numeric_columns] = df[numeric_columns].fillna(0)

print("Missing values after cleaning:")
print(df.isnull().sum())

Missing values after cleaning:
content_hash_id           0
total_impressions         0
total_clicks              0
avg_position              0
total_pageviews           0
total_sessions            0
total_users               0
total_engaged_sessions    0
total_engagement_time     0
organic_sessions          0
direct_sessions           0
referral_sessions         0
social_sessions           0
ai_sessions               0
total_scroll_events       0
dtype: int64


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

features = [
    "total_impressions",
    "total_clicks",
    "avg_position",
    "total_pageviews",
    "total_sessions",
    "total_users",
    "total_engaged_sessions",
    "total_engagement_time",
    "organic_sessions",
    "direct_sessions",
    "referral_sessions",
    "social_sessions",
    "ai_sessions",
    "total_scroll_events"
]

X = df[features]

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

kmeans = KMeans(
    n_clusters=2,
    random_state=42,
    n_init=10
)

df["cluster"] = kmeans.fit_predict(X_scaled)

print(df["cluster"].value_counts())

cluster
0    49999
1        1
Name: count, dtype: int64


In [ ]:
cluster_summary = df.groupby("cluster")[
    [
        "total_impressions",
        "total_clicks",
        "total_pageviews",
        "organic_sessions",
        "total_scroll_events"
    ]
].mean()

print(cluster_summary)

         total_impressions  total_clicks  total_pageviews  organic_sessions  \
cluster                                                                       
0             14795.293646     52.366187         57.02112         25.634093   
1            194250.000000    552.000000     108311.00000        125.000000   

         total_scroll_events  
cluster                       
0                   4.464729  
1                9769.000000  


In [ ]:
normalized = (
    cluster_summary - cluster_summary.min()
) / (
    cluster_summary.max() - cluster_summary.min()
)

cluster_score = normalized.mean(axis=1)

higher_cluster = cluster_score.idxmax()
lower_cluster = cluster_score.idxmin()

print("Higher-Performance Cluster:", higher_cluster)
print("Lower-Performance Cluster:", lower_cluster)

Higher-Performance Cluster: 1
Lower-Performance Cluster: 0


In [ ]:
!pip install -q duckdb pandas numpy scikit-learn

In [ ]:
import duckdb
import pandas as pd
import numpy as np

con = duckdb.connect()

con.execute("""
CREATE SECRET (
    TYPE huggingface,
    TOKEN 'YOUR_HF_READ_TOKEN'
)
""")

rel = "hf://datasets/FlyRank/internship-warehouse"

print("Connected to FlyRank dataset")

Connected to FlyRank dataset


In [ ]:
df["avg_position"] = df["avg_position"].fillna(
    df["avg_position"].median()
)

numeric_columns = df.select_dtypes(
    include=[np.number]
).columns

df[numeric_columns] = df[numeric_columns].fillna(0)

print(df.isnull().sum())

content_hash_id           0
total_impressions         0
total_clicks              0
avg_position              0
total_pageviews           0
total_sessions            0
total_users               0
total_engaged_sessions    0
total_engagement_time     0
organic_sessions          0
direct_sessions           0
referral_sessions         0
social_sessions           0
ai_sessions               0
total_scroll_events       0
cluster                   0
dtype: int64


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

features = [
    "total_impressions",
    "total_clicks",
    "avg_position",
    "total_pageviews",
    "total_sessions",
    "total_users",
    "total_engaged_sessions",
    "total_engagement_time",
    "organic_sessions",
    "direct_sessions",
    "referral_sessions",
    "social_sessions",
    "ai_sessions",
    "total_scroll_events"
]

X = df[features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(
    n_clusters=2,
    random_state=42,
    n_init=10
)

df["cluster"] = kmeans.fit_predict(X_scaled)

print(df["cluster"].value_counts())

cluster
0    49999
1        1
Name: count, dtype: int64


In [ ]:
cluster_summary = df.groupby("cluster")[
    [
        "total_impressions",
        "total_clicks",
        "total_pageviews",
        "organic_sessions",
        "total_scroll_events"
    ]
].mean()

normalized = (
    cluster_summary - cluster_summary.min()
) / (
    cluster_summary.max() - cluster_summary.min()
)

cluster_score = normalized.mean(axis=1)

higher_cluster = cluster_score.idxmax()
lower_cluster = cluster_score.idxmin()

df["content_archetype"] = df["cluster"].map({
    higher_cluster: "Higher-Performance Content",
    lower_cluster: "Lower-Performance Content"
})

print(df["content_archetype"].value_counts())

content_archetype
Lower-Performance Content     49999
Higher-Performance Content        1
Name: count, dtype: int64


In [ ]:
def assign_action(row):

    if row["content_archetype"] == "Lower-Performance Content":
        return pd.Series([
            "High",
            "Investigate and improve content performance",
            "LOW_PERFORMANCE"
        ])

    return pd.Series([
        "Low",
        "Monitor and study successful characteristics",
        "HIGH_PERFORMANCE"
    ])


df[["priority", "recommended_action", "reason_code"]] = df.apply(
    assign_action,
    axis=1
)

df[[
    "content_hash_id",
    "content_archetype",
    "priority",
    "recommended_action",
    "reason_code"
]].head()

,content_hash_id,content_archetype,priority,recommended_action,reason_code
0,content_d5479697a0828b33,Lower-Performance Content,High,Investigate and improve content performance,LOW_PERFORMANCE
1,content_d7daac9d28863fda,Lower-Performance Content,High,Investigate and improve content performance,LOW_PERFORMANCE
2,content_b313be479d6d3707,Lower-Performance Content,High,Investigate and improve content performance,LOW_PERFORMANCE
3,content_5175438fecb054a4,Lower-Performance Content,High,Investigate and improve content performance,LOW_PERFORMANCE
4,content_3a204f16288e29ed,Lower-Performance Content,High,Investigate and improve content performance,LOW_PERFORMANCE


In [ ]:
action_mapping = pd.DataFrame({
    "content_archetype": [
        "Higher-Performance Content",
        "Lower-Performance Content"
    ],

    "priority": [
        "Low",
        "High"
    ],

    "recommended_action": [
        "Monitor performance and study characteristics associated with stronger observed results",
        "Prioritize for investigation and review search visibility, traffic, relevance, readability, and engagement"
    ],

    "reason_code": [
        "HIGH_PERFORMANCE",
        "LOW_PERFORMANCE"
    ]
})

action_mapping

,content_archetype,priority,recommended_action,reason_code
0,Higher-Performance Content,Low,Monitor performance and study characteristics ...,HIGH_PERFORMANCE
1,Lower-Performance Content,High,Prioritize for investigation and review search...,LOW_PERFORMANCE


In [ ]:
human_review_rules = pd.DataFrame({
    "situation": [
        "Lower-performing content with unusual or suspicious data",
        "Content with missing business context",
        "Content with a major recent performance change",
        "Higher-performing content used as a benchmark"
    ],

    "action": [
        "Require human data review before taking action",
        "Require human review before changing content",
        "Investigate manually before taking action",
        "Do not automatically copy or modify content"
    ],

    "decision": [
        "NO-GO",
        "NO-GO",
        "NO-GO",
        "MONITOR ONLY"
    ]
})

human_review_rules

,situation,action,decision
0,Lower-performing content with unusual or suspi...,Require human data review before taking action,NO-GO
1,Content with missing business context,Require human review before changing content,NO-GO
2,Content with a major recent performance change,Investigate manually before taking action,NO-GO
3,Higher-performing content used as a benchmark,Do not automatically copy or modify content,MONITOR ONLY


In [ ]:
priority_order = {
    "High": 1,
    "Low": 2
}

df["priority_rank"] = df["priority"].map(priority_order)

action_queue = df.sort_values(
    by="priority_rank",
    ascending=True
).copy()

action_queue = action_queue[
    [
        "content_hash_id",
        "cluster",
        "content_archetype",
        "priority",
        "recommended_action",
        "reason_code"
    ]
]

print("Ranked Content Action Queue")
print("Total content items:", len(action_queue))

action_queue.head(20)

Ranked Content Action Queue
Total content items: 50000


,content_hash_id,cluster,content_archetype,priority,recommended_action,reason_code
33338,content_b21d4fe87049845a,0,Lower-Performance Content,High,Investigate and improve content performance,LOW_PERFORMANCE
33326,content_3e0e9af25462b281,0,Lower-Performance Content,High,Investigate and improve content performance,LOW_PERFORMANCE
33327,content_501c0444d4e9dbe1,0,Lower-Performance Content,High,Investigate and improve content performance,LOW_PERFORMANCE
33328,content_07518923afb05c3d,0,Lower-Performance Content,High,Investigate and improve content performance,LOW_PERFORMANCE
33329,content_d0e2557be31e965a,0,Lower-Performance Content,High,Investigate and improve content performance,LOW_PERFORMANCE
33330,content_a502817ff999caf1,0,Lower-Performance Content,High,Investigate and improve content performance,LOW_PERFORMANCE
33331,content_31baa56b7eb9f992,0,Lower-Performance Content,High,Investigate and improve content performance,LOW_PERFORMANCE
33332,content_29873d436b6fcb02,0,Lower-Performance Content,High,Investigate and improve content performance,LOW_PERFORMANCE
33333,content_41bd8f0e82132f02,0,Lower-Performance Content,High,Investigate and improve content performance,LOW_PERFORMANCE
33334,content_293177d40ec3a91e,0,Lower-Performance Content,High,Investigate and improve content performance,LOW_PERFORMANCE


In [ ]:
monitoring_triggers = pd.DataFrame({
    "metric": [
        "total_impressions",
        "total_clicks",
        "avg_position",
        "total_pageviews"
    ],
    "threshold_type": [
        "drop",
        "drop",
        "increase",
        "drop"
    ],
    "threshold_value": [
        0.1,  # 10% drop
        0.15, # 15% drop
        0.05, # 5% increase (e.g., avg_position decreasing means better ranking)
        0.1  # 10% drop
    ],
    "action": [
        "Investigate unexpected performance drop",
        "Investigate unexpected performance drop",
        "Investigate unexpected rank improvement",
        "Investigate unexpected performance drop"
    ]
})

print("Monitoring Triggers:")
display(monitoring_triggers)

Monitoring Triggers:


,metric,threshold_type,threshold_value,action
0,total_impressions,drop,0.10,Investigate unexpected performance drop
1,total_clicks,drop,0.15,Investigate unexpected performance drop
2,avg_position,increase,0.05,Investigate unexpected rank improvement
3,total_pageviews,drop,0.10,Investigate unexpected performance drop


In [ ]:
import os

os.makedirs("/content/outputs", exist_ok=True)

# 1. Ranked action queue
action_queue.to_csv(
    "/content/outputs/content_action_queue.csv",
    index=False
)

# 2. Archetype-to-action mapping
action_mapping.to_csv(
    "/content/outputs/archetype_action_mapping.csv",
    index=False
)

# 3. Human review rules
human_review_rules.to_csv(
    "/content/outputs/human_review_rules.csv",
    index=False
)

# 4. Monitoring and retraining triggers
monitoring_triggers.to_csv(
    "/content/outputs/monitoring_retraining_triggers.csv",
    index=False
)

print("All files exported successfully!")
print("\nOutput files:")
print("1. content_action_queue.csv")
print("2. archetype_action_mapping.csv")
print("3. human_review_rules.csv")
print("4. monitoring_retraining_triggers.csv")

All files exported successfully!

Output files:
1. content_action_queue.csv
2. archetype_action_mapping.csv
3. human_review_rules.csv
4. monitoring_retraining_triggers.csv
